# CPDE-7 Configurable Extraction Testing

This notebook tests the new `extract_dimensions()` method with two strategies:

1. **Per-Dimension Strategy** (`strategy="per_dimension"`)
   - Separate LLM call for each dimension
   - Uses static prompts from `batch_prompts.py`
   - Can run sequentially or in parallel
   - **No Proteas**

2. **Combined Strategy** (`strategy="combined"`)
   - Single LLM call for all selected dimensions
   - Uses **Proteas** to dynamically build prompts
   - More efficient (fewer API calls)

Both strategies return `BatchProfilingDataExtractionResult` with consistent semantics:
- `None` = dimension was not requested
- `has_content=False` = requested but nothing found
- `has_content=True` = requested and data found

In [ ]:
# Load environment variables
from dotenv import load_dotenv
import os

# Load from project root .env file (6 levels up now)
load_dotenv("../../../../../../.env", override=True)

# Verify API key is loaded
print(f"OPENAI_API_KEY loaded: {'Yes' if os.getenv('OPENAI_API_KEY') else 'No'}")

In [ ]:
# Import the CPDE7LLMService and related modules
import time
import json

from chatforge.services.profiling_data_extraction import (
    CPDE7LLMService,
    CPF7_DIMENSIONS,
    BatchProfilingDataExtractionResult,
)
from chatforge.services.profiling_data_extraction.cpde7.prompts import (
    build_prompt,
    build_output_model,
    DIMENSION_NAMES,
)

print(f"Available dimensions: {CPF7_DIMENSIONS}")
print(f"DIMENSION_NAMES: {DIMENSION_NAMES}")

In [3]:
# Initialize the service
cpde_service = CPDE7LLMService(
    provider="openai",
    model_name="gpt-4o-mini",
    temperature=0,
)

print(f"Service initialized: {cpde_service.model_info}")

Service initialized: openai/gpt-4o-mini


In [4]:
# Helper function to display pde_result
def display_pde_result(pde_result: BatchProfilingDataExtractionResult, title: str = "Results"):
    """Pretty print BatchProfilingDataExtractionResult."""
    print(f"\n{'='*60}")
    print(f"{title}")
    print(f"{'='*60}")
    
    total = 0
    for dim_name in CPF7_DIMENSIONS:
        dim_result = getattr(pde_result, dim_name, None)
        if dim_result is None:
            print(f"  {dim_name:25} : None (not requested)")
        else:
            count = len(dim_result.items)
            total += count
            status = "has_content" if dim_result.has_content else "empty"
            print(f"  {dim_name:25} : {count:3} items ({status})")
    
    print(f"\n  {'TOTAL':25} : {total:3} items")


def display_dimension_detail(pde_result: BatchProfilingDataExtractionResult, dim_name: str):
    """Show detailed items for a specific dimension."""
    dim_result = getattr(pde_result, dim_name, None)
    
    print(f"\n{'='*60}")
    print(f"{dim_name.upper()} - Detailed View")
    print(f"{'='*60}")
    
    if dim_result is None:
        print("Not requested")
        return
    
    print(f"Has content: {dim_result.has_content}")
    print(f"Items: {len(dim_result.items)}")
    
    for i, item in enumerate(dim_result.items, 1):
        print(f"\n--- Item {i} ---")
        print(f"  Source: {item.source_message_id}")
        print(f"  Quote: \"{item.source_quote}\"")
        for field_name, field_value in item.model_dump().items():
            if field_name not in ['source_message_id', 'source_quote'] and field_value is not None:
                print(f"  {field_name}: {field_value}")

## Test Messages

A rich set of messages containing data for all 7 dimensions.

In [5]:
# Comprehensive test messages covering all 7 dimensions
test_messages = """
Message ID: msg_001
Content: Hi! I'm a 34-year-old software engineer living in Seattle. I'm also diabetic.

Message ID: msg_002
Content: I think remote work is the future, but companies need better async communication tools.

Message ID: msg_003
Content: I always code better at night and I can't work without my noise-canceling headphones.

Message ID: msg_004
Content: I really want to transition into AI/ML and I hope to lead a team someday.

Message ID: msg_005
Content: I lived in Tokyo for 3 years after college and it completely changed my worldview.

Message ID: msg_006
Content: I'm currently interviewing at Google and also dealing with a kitchen renovation that's driving me crazy.

Message ID: msg_007
Content: My mentor Sarah from my previous job at Microsoft still helps me weekly. My dog Max keeps me company while I work.
"""

print("Test messages loaded:")
print(test_messages)

Test messages loaded:

Message ID: msg_001
Content: Hi! I'm a 34-year-old software engineer living in Seattle. I'm also diabetic.

Message ID: msg_002
Content: I think remote work is the future, but companies need better async communication tools.

Message ID: msg_003
Content: I always code better at night and I can't work without my noise-canceling headphones.

Message ID: msg_004
Content: I really want to transition into AI/ML and I hope to lead a team someday.

Message ID: msg_005
Content: I lived in Tokyo for 3 years after college and it completely changed my worldview.

Message ID: msg_006
Content: I'm currently interviewing at Google and also dealing with a kitchen renovation that's driving me crazy.

Message ID: msg_007
Content: My mentor Sarah from my previous job at Microsoft still helps me weekly. My dog Max keeps me company while I work.



---
## Strategy 1: Per-Dimension (Non-Proteas)

Makes separate LLM calls for each dimension using static prompts.

In [6]:
# Per-dimension strategy - SEQUENTIAL
print("Testing per-dimension strategy (SEQUENTIAL)...")
print("Extracting: core_identity, events, entities_relationships")
print("(3 separate LLM calls, one after another)\n")

start = time.time()
pde_result_seq = await cpde_service.extract_dimensions(
    messages=test_messages,
    dimensions=["core_identity", "events", "entities_relationships"],
    strategy="per_dimension",
    parallel=False
)
elapsed_seq = time.time() - start

print(f"Completed in {elapsed_seq:.2f}s")
display_pde_result(pde_result_seq, "Per-Dimension Sequential Results")

Testing per-dimension strategy (SEQUENTIAL)...
Extracting: core_identity, events, entities_relationships
(3 separate LLM calls, one after another)

Completed in 22.06s

Per-Dimension Sequential Results
  core_identity             :   4 items (has_content)
  opinions_views            : None (not requested)
  preferences_patterns      : None (not requested)
  desires_needs             : None (not requested)
  life_narrative            : None (not requested)
  events                    :   2 items (has_content)
  entities_relationships    :   4 items (has_content)

  TOTAL                     :  10 items


In [7]:
# Per-dimension strategy - PARALLEL
print("Testing per-dimension strategy (PARALLEL)...")
print("Extracting: core_identity, events, entities_relationships")
print("(3 LLM calls running concurrently)\n")

start = time.time()
pde_result_par = await cpde_service.extract_dimensions(
    messages=test_messages,
    dimensions=["core_identity", "events", "entities_relationships"],
    strategy="per_dimension",
    parallel=True
)
elapsed_par = time.time() - start

print(f"Completed in {elapsed_par:.2f}s")
print(f"Speedup vs sequential: {elapsed_seq/elapsed_par:.2f}x")
display_pde_result(pde_result_par, "Per-Dimension Parallel Results")

Testing per-dimension strategy (PARALLEL)...
Extracting: core_identity, events, entities_relationships
(3 LLM calls running concurrently)

Completed in 11.15s
Speedup vs sequential: 1.98x

Per-Dimension Parallel Results
  core_identity             :   4 items (has_content)
  opinions_views            : None (not requested)
  preferences_patterns      : None (not requested)
  desires_needs             : None (not requested)
  life_narrative            : None (not requested)
  events                    :   2 items (has_content)
  entities_relationships    :   4 items (has_content)

  TOTAL                     :  10 items


---
## Strategy 2: Combined (Proteas)

Makes a single LLM call with a dynamically-built prompt using Proteas.

In [8]:
# Combined strategy - using Proteas
print("Testing combined strategy (PROTEAS)...")
print("Extracting: core_identity, events, entities_relationships")
print("(1 LLM call with dynamically-built prompt)\n")

start = time.time()
pde_result_combined = await cpde_service.extract_dimensions(
    messages=test_messages,
    dimensions=["core_identity", "events", "entities_relationships"],
    strategy="combined"
)
elapsed_combined = time.time() - start

print(f"Completed in {elapsed_combined:.2f}s")
print(f"Speedup vs sequential per-dim: {elapsed_seq/elapsed_combined:.2f}x")
print(f"Speedup vs parallel per-dim: {elapsed_par/elapsed_combined:.2f}x")
display_pde_result(pde_result_combined, "Combined (Proteas) Results")

Testing combined strategy (PROTEAS)...
Extracting: core_identity, events, entities_relationships
(1 LLM call with dynamically-built prompt)

Completed in 16.06s
Speedup vs sequential per-dim: 1.37x
Speedup vs parallel per-dim: 0.69x

Combined (Proteas) Results
  core_identity             :   4 items (has_content)
  opinions_views            : None (not requested)
  preferences_patterns      : None (not requested)
  desires_needs             : None (not requested)
  life_narrative            : None (not requested)
  events                    :   2 items (has_content)
  entities_relationships    :   2 items (has_content)

  TOTAL                     :   8 items


In [9]:
# Show the Proteas-built prompt
print("Proteas-built prompt for [core_identity, events, entities_relationships]:")
print("="*60)
prompt = build_prompt(
    dimensions=["core_identity", "events", "entities_relationships"],
    messages="<messages would go here>"
)
print(prompt[:2000] + "\n...\n[truncated]")

Proteas-built prompt for [core_identity, events, entities_relationships]:
Your job is to extract profiling data from the given batch of chat messages.
Extract information across the specified dimensions.

CRITICAL RULES:
- Do NOT infer or assume information not explicitly stated
- Maintain dimensional boundaries - put data in the correct dimension
- YOU must not force a meaning. IT IS OKAY TO NOT FIND DATA
- Each extracted item MUST include source attribution (source_message_id and source_quote)

DIMENSION: CORE IDENTITY

### What It Is
- What someone IS (roles, attributes, states, affiliations)
- Stable characteristics that define them
- Permanent or long-lasting aspects of who they are
- Examples: age, profession, location, physical attributes, chronic conditions, personality traits

### What It Is Not
- What they THINK → goes to Opinions & Views
- What they WANT → goes to Desires, Wishes, Hopes & Needs
- What they DID → goes to Events or Life Narrative
- What they PREFER → goes to P

In [10]:
# Show the dynamic output model
print("Dynamic output model for [core_identity, events]:")
print("="*60)
model = build_output_model(["core_identity", "events"])
print(f"Model name: {model.__name__}")
print(f"Fields: {list(model.model_fields.keys())}")
print(f"\nJSON Schema:")
print(json.dumps(model.model_json_schema(), indent=2))

Dynamic output model for [core_identity, events]:
Model name: DynamicExtractionOutput
Fields: ['core_identity', 'events']

JSON Schema:
{
  "$defs": {
    "BatchCoreIdentityItem": {
      "description": "A single identity fact with source attribution.",
      "properties": {
        "source_message_id": {
          "description": "The message ID where this was extracted from",
          "title": "Source Message Id",
          "type": "string"
        },
        "source_quote": {
          "description": "The exact quote from the message",
          "title": "Source Quote",
          "type": "string"
        },
        "aspect": {
          "description": "Category of identity marker (e.g., age, profession, location, physical attribute, role, affiliation, condition, personality)",
          "title": "Aspect",
          "type": "string"
        },
        "state_value": {
          "description": "The actual value/state",
          "title": "State Value",
          "type": "string"
     

---
## Combined Strategy: All 7 Dimensions

When all 7 dimensions are requested with combined strategy, it uses the optimized `extract_all_7()` internally.

In [11]:
# Combined strategy with ALL 7 dimensions
print("Testing combined strategy with ALL 7 dimensions...")
print("(Uses optimized extract_all_7() internally)\n")

start = time.time()
pde_result_all7 = await cpde_service.extract_dimensions(
    messages=test_messages,
    dimensions=CPF7_DIMENSIONS,  # All 7 dimensions
    strategy="combined"
)
elapsed_all7 = time.time() - start

print(f"Completed in {elapsed_all7:.2f}s")
display_pde_result(pde_result_all7, "All 7 Dimensions (Combined Strategy)")

Testing combined strategy with ALL 7 dimensions...
(Uses optimized extract_all_7() internally)

Completed in 29.06s

All 7 Dimensions (Combined Strategy)
  core_identity             :   4 items (has_content)
  opinions_views            :   1 items (has_content)
  preferences_patterns      :   2 items (has_content)
  desires_needs             :   2 items (has_content)
  life_narrative            :   1 items (has_content)
  events                    :   2 items (has_content)
  entities_relationships    :   2 items (has_content)

  TOTAL                     :  14 items


---
## Comparing Results: Per-Dimension vs Combined

Let's compare the extraction results between the two strategies.

In [12]:
# Compare core_identity results
print("Comparing core_identity results between strategies:")
print("="*60)

print("\nPER-DIMENSION (parallel):")
display_dimension_detail(pde_result_par, "core_identity")

print("\nCOMBINED (Proteas):")
display_dimension_detail(pde_result_combined, "core_identity")

Comparing core_identity results between strategies:

PER-DIMENSION (parallel):

CORE_IDENTITY - Detailed View
Has content: True
Items: 4

--- Item 1 ---
  Source: msg_001
  Quote: "I'm a 34-year-old software engineer living in Seattle."
  aspect: age
  state_value: 34 years old

--- Item 2 ---
  Source: msg_001
  Quote: "I'm a 34-year-old software engineer living in Seattle."
  aspect: profession
  state_value: software engineer

--- Item 3 ---
  Source: msg_001
  Quote: "I'm a 34-year-old software engineer living in Seattle."
  aspect: location
  state_value: Seattle

--- Item 4 ---
  Source: msg_001
  Quote: "I'm also diabetic."
  aspect: medical_condition
  state_value: diabetic

COMBINED (Proteas):

CORE_IDENTITY - Detailed View
Has content: True
Items: 4

--- Item 1 ---
  Source: msg_001
  Quote: "I'm a 34-year-old software engineer living in Seattle."
  aspect: age
  state_value: 34 years old

--- Item 2 ---
  Source: msg_001
  Quote: "I'm a 34-year-old software engineer living i

In [13]:
# Compare events results
print("Comparing events results between strategies:")
print("="*60)

print("\nPER-DIMENSION (parallel):")
display_dimension_detail(pde_result_par, "events")

print("\nCOMBINED (Proteas):")
display_dimension_detail(pde_result_combined, "events")

Comparing events results between strategies:

PER-DIMENSION (parallel):

EVENTS - Detailed View
Has content: True
Items: 2

--- Item 1 ---
  Source: msg_006
  Quote: "I'm currently interviewing at Google and also dealing with a kitchen renovation that's driving me crazy."
  event: job interview
  involvement: candidate
  temporal: currently
  entities_involved: ['Google']

--- Item 2 ---
  Source: msg_006
  Quote: "I'm currently interviewing at Google and also dealing with a kitchen renovation that's driving me crazy."
  event: kitchen renovation
  involvement: homeowner
  temporal: ongoing
  outcome: driving me crazy

COMBINED (Proteas):

EVENTS - Detailed View
Has content: True
Items: 2

--- Item 1 ---
  Source: msg_006
  Quote: "I'm currently interviewing at Google and also dealing with a kitchen renovation that's driving me crazy."
  event: job interview
  involvement: candidate
  temporal: currently
  entities_involved: ['Google']

--- Item 2 ---
  Source: msg_006
  Quote: "I'm cu

---
## Testing Single Dimension Extraction

In [ ]:
# Single dimension with combined strategy
print("Testing single dimension extraction (combined strategy)...")
print("Extracting only: opinions_views\n")

start = time.time()
pde_result_single = await cpde_service.extract_dimensions(
    messages=test_messages,
    dimensions=["opinions_views"],
    strategy="combined"
)
elapsed_single = time.time() - start

print(f"Completed in {elapsed_single:.2f}s")
display_pde_result(pde_result_single, "Single Dimension Results")
display_dimension_detail(pde_result_single, "opinions_views")

---
## Validation Testing

Test error handling for invalid inputs.

In [14]:
# Test empty dimensions list
print("Testing empty dimensions list...")
try:
    await cpde_service.extract_dimensions(
        messages=test_messages,
        dimensions=[],
        strategy="combined"
    )
    print("ERROR: Should have raised ValueError!")
except ValueError as e:
    print(f"Correctly raised ValueError: {e}")

Testing empty dimensions list...
Correctly raised ValueError: At least one dimension must be specified


In [15]:
# Test invalid dimension name
print("Testing invalid dimension name...")
try:
    await cpde_service.extract_dimensions(
        messages=test_messages,
        dimensions=["invalid_dimension"],
        strategy="combined"
    )
    print("ERROR: Should have raised ValueError!")
except ValueError as e:
    print(f"Correctly raised ValueError: {e}")

Testing invalid dimension name...
Correctly raised ValueError: Invalid dimension: invalid_dimension. Valid: ['core_identity', 'opinions_views', 'preferences_patterns', 'desires_needs', 'life_narrative', 'events', 'entities_relationships']


---
## Result Semantics: None vs has_content=False

Demonstrating the difference between:
- `None` = dimension was not requested
- `has_content=False` = dimension was requested but nothing was found

In [16]:
# Edge case messages with minimal extractable data
edge_case_messages = """
Message ID: edge_001
Content: This coffee is cold and the meeting was pointless.

Message ID: edge_002
Content: I had lunch and then went to the store.

Message ID: edge_003
Content: Apple announced a new iPhone yesterday.
"""

print("Edge case messages (should extract minimal data):")
print(edge_case_messages)

Edge case messages (should extract minimal data):

Message ID: edge_001
Content: This coffee is cold and the meeting was pointless.

Message ID: edge_002
Content: I had lunch and then went to the store.

Message ID: edge_003
Content: Apple announced a new iPhone yesterday.



In [17]:
# Extract with combined strategy
print("Extracting from edge case messages...")
print("Extracting: core_identity, opinions_views, desires_needs\n")

pde_result_edge = await cpde_service.extract_dimensions(
    messages=edge_case_messages,
    dimensions=["core_identity", "opinions_views", "desires_needs"],
    strategy="combined"
)

print("Result semantics:")
print("="*60)

for dim_name in CPF7_DIMENSIONS:
    dim_result = getattr(pde_result_edge, dim_name, None)
    if dim_result is None:
        print(f"  {dim_name:25} : None (NOT requested)")
    elif not dim_result.has_content:
        print(f"  {dim_name:25} : has_content=False (requested, nothing found)")
    else:
        print(f"  {dim_name:25} : has_content=True ({len(dim_result.items)} items found)")

Extracting from edge case messages...
Extracting: core_identity, opinions_views, desires_needs

Result semantics:
  core_identity             : has_content=False (requested, nothing found)
  opinions_views            : has_content=False (requested, nothing found)
  preferences_patterns      : None (NOT requested)
  desires_needs             : has_content=False (requested, nothing found)
  life_narrative            : None (NOT requested)
  events                    : None (NOT requested)
  entities_relationships    : None (NOT requested)


---
## Performance Summary

In [18]:
# Performance comparison summary
print("\n" + "="*60)
print("PERFORMANCE SUMMARY (3 dimensions: core_identity, events, entities)")
print("="*60)
print(f"\n  Per-Dimension (Sequential) : {elapsed_seq:.2f}s  (3 LLM calls)")
print(f"  Per-Dimension (Parallel)   : {elapsed_par:.2f}s  (3 LLM calls concurrent)")
print(f"  Combined (Proteas)         : {elapsed_combined:.2f}s  (1 LLM call)")
print(f"\n  Combined speedup vs sequential: {elapsed_seq/elapsed_combined:.2f}x")
print(f"  Combined speedup vs parallel:   {elapsed_par/elapsed_combined:.2f}x")
print(f"\n  All 7 dimensions (combined): {elapsed_all7:.2f}s  (1 LLM call)")


PERFORMANCE SUMMARY (3 dimensions: core_identity, events, entities)

  Per-Dimension (Sequential) : 22.06s  (3 LLM calls)
  Per-Dimension (Parallel)   : 11.15s  (3 LLM calls concurrent)
  Combined (Proteas)         : 16.06s  (1 LLM call)

  Combined speedup vs sequential: 1.37x
  Combined speedup vs parallel:   0.69x

  All 7 dimensions (combined): 29.06s  (1 LLM call)


---
## Raw JSON Output

In [ ]:
# Show raw JSON for combined strategy result
print("Raw JSON output (Combined Strategy - 3 dimensions):")
print(json.dumps(pde_result_combined.model_dump(), indent=2))